In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/caf-dataset/CAF_sensors.dbf
/kaggle/input/caf-dataset/Hourly/Hourly/CAF141.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF019.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF209.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF007.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF397.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF237.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF205.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF125.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF139.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF095.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF201.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF215.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF079.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF231.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF275.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF003.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF197.txt
/kaggle/input/caf-dataset/Hourly/Hourly/CAF351.txt
/kaggle/input/caf-dataset/Hourly/Hourly/

In [2]:
import numpy as np
import pandas as pd
import os

from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

np.random.seed(42)


In [6]:
# Input paths (Kaggle)
CAF_DATASET_PATH = "/kaggle/input/caf-dataset"
BASELINE_PATH = "/kaggle/input/baseline-artifacts"

HOURLY_PATH = os.path.join(CAF_DATASET_PATH, "Hourly", "Hourly")

print("Hourly files:", len(os.listdir(HOURLY_PATH)))


Hourly files: 42


In [7]:
def rmse(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(mean_squared_error(a[mask], b[mask]))


In [8]:
df_wide = pd.read_parquet(
    os.path.join(BASELINE_PATH, "df_wide.parquet")
)

df_pred_spatial = pd.read_parquet(
    os.path.join(BASELINE_PATH, "df_pred.parquet")
)

df_resid_spatial = pd.read_parquet(
    os.path.join(BASELINE_PATH, "df_resid.parquet")
)

df_wide.shape, df_pred_spatial.shape


((66705, 10), (66705, 10))

In [9]:
def temporal_model(series, span=24*7):
    return series.ewm(span=span, adjust=False).mean()

df_pred_temporal = pd.DataFrame(index=df_wide.index)

for sensor in df_wide.columns:
    df_pred_temporal[sensor] = temporal_model(df_wide[sensor])


In [10]:
r_spatial = df_wide - df_pred_spatial
r_temporal = df_wide - df_pred_temporal


In [11]:
def build_feature_matrix(
    df_wide,
    r_spatial,
    r_temporal,
    window=24
):
    rows = []

    for sensor in df_wide.columns:
        df_feat = pd.DataFrame({
            "r_spatial": r_spatial[sensor],
            "r_temporal": r_temporal[sensor],
            "delta": df_wide[sensor].diff(),
            "rolling_std": df_wide[sensor].rolling(window).std(),
            "is_nan": df_wide[sensor].isna().astype(int),
        })

        df_feat["sensor"] = sensor
        df_feat["timestamp"] = df_wide.index
        rows.append(df_feat)

    return pd.concat(rows).dropna()


In [12]:
X = build_feature_matrix(
    df_wide,
    r_spatial,
    r_temporal
)

X.head(), X.shape


(                     r_spatial  r_temporal  delta  rolling_std  is_nan  \
 timestamp                                                                
 2009-05-22 15:00:00    -0.0772   -0.000224  0.000     0.000955       0   
 2009-05-22 16:00:00    -0.0774   -0.000222  0.000     0.000955       0   
 2009-05-22 17:00:00    -0.0766    0.000769  0.001     0.000955       0   
 2009-05-22 18:00:00    -0.0770    0.000760  0.000     0.000929       0   
 2009-05-22 19:00:00    -0.0768    0.001739  0.001     0.000929       0   
 
                      sensor           timestamp  
 timestamp                                        
 2009-05-22 15:00:00  CAF003 2009-05-22 15:00:00  
 2009-05-22 16:00:00  CAF003 2009-05-22 16:00:00  
 2009-05-22 17:00:00  CAF003 2009-05-22 17:00:00  
 2009-05-22 18:00:00  CAF003 2009-05-22 18:00:00  
 2009-05-22 19:00:00  CAF003 2009-05-22 19:00:00  ,
 (393381, 7))

In [13]:
features = [
    "r_spatial",
    "r_temporal",
    "delta",
    "rolling_std",
    "is_nan"
]

model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42
)

X["fault"] = model.fit_predict(X[features])
X["fault"] = (X["fault"] == -1).astype(int)


In [14]:
fault_mask = (
    X.pivot(
        index="timestamp",
        columns="sensor",
        values="fault"
    )
    .reindex(df_wide.index)
    .fillna(0)
    .astype(bool)
)

fault_mask.head()


sensor,CAF003,CAF007,CAF009,CAF019,CAF031,CAF033,CAF035,CAF061,CAF067,CAF075
timestamp,,,,,,,,,,
2007-05-07 17:00:00,False,False,False,False,False,False,False,False,False,False
2007-05-07 18:00:00,False,False,False,False,False,False,False,False,False,False
2007-05-07 19:00:00,False,False,False,False,False,False,False,False,False,False
2007-05-07 20:00:00,False,False,False,False,False,False,False,False,False,False
2007-05-07 21:00:00,False,False,False,False,False,False,False,False,False,False


In [15]:
df_comp = df_wide.copy()

for sensor in df_wide.columns:
    df_comp.loc[fault_mask[sensor], sensor] = (
        0.5 * df_pred_spatial[sensor] +
        0.5 * df_pred_temporal[sensor]
    )


In [16]:
results = []

for sensor in df_wide.columns:
    mask = fault_mask[sensor]

    rmse_faulty = rmse(
        df_pred_spatial[sensor][mask],
        df_wide[sensor][mask]
    )

    rmse_comp = rmse(
        df_pred_spatial[sensor][mask],
        df_comp[sensor][mask]
    )

    results.append({
        "sensor": sensor,
        "fault_rate": mask.mean(),
        "rmse_faulty": rmse_faulty,
        "rmse_compensated": rmse_comp,
        "recovery_ratio": (
            1 - rmse_comp / rmse_faulty
            if rmse_faulty and not np.isnan(rmse_faulty)
            else np.nan
        )
    })

results_df = pd.DataFrame(results)
results_df


,sensor,fault_rate,rmse_faulty,rmse_compensated,recovery_ratio
0,CAF003,0.025320,0.031232,0.013599,0.564598
1,CAF007,0.033356,0.032486,0.023928,0.263421
2,CAF009,0.012623,0.033722,0.018148,0.461827
3,CAF019,0.012683,0.025847,0.020995,0.187699
4,CAF031,0.034090,0.067307,0.020547,0.694721
5,CAF033,0.018874,0.079313,0.045219,0.429869
6,CAF035,0.032666,0.056259,0.018238,0.675813
7,CAF061,0.071539,0.162900,0.081655,0.498745
8,CAF067,0.034525,0.032004,0.019902,0.378131
9,CAF075,0.019159,0.029786,0.020645,0.306888


In [17]:
OUT_PATH = "/kaggle/working/outputs"
os.makedirs(OUT_PATH, exist_ok=True)

results_df.to_csv(f"{OUT_PATH}/results_summary.csv", index=False)
fault_mask.to_parquet(f"{OUT_PATH}/fault_mask.parquet")
df_comp.to_parquet(f"{OUT_PATH}/df_compensated.parquet")


In [18]:
import shutil

shutil.make_archive(
    "/kaggle/working/CAF_spatiotemporal_outputs",
    "zip",
    OUT_PATH
)


'/kaggle/working/CAF_spatiotemporal_outputs.zip'